In [ ]:
# Problem Statement :

# Fake News Classification with The Help Of Natural Language Processing Technique.

# Fake news detection is a hot topic in the field of natural language processing. We consume news through several mediums throughout the day in our daily routine, but sometimes it becomes difficult to decide which one is fake and which one is authentic. Our job is to create a model which predicts whether a given news is real or fake.

# Project Flow:
# 1. Problem Statement
# 2. Data Gathering
# 3. Data Preprocessing : Here we perform some operation on data
#    A. Tokenization
#    B. Lower Case
#    C. Stopwords
#    D. Lemmatization / Stemming
# 4. Vectorization (Convert Text data into the Vector):
#    A. Bag Of Words (CountVectorizer)
#    B. TF-IDF
# 5. Model Building :
#    A. Model Object Initialization
#    B. Train and Test Model
# 6. Model Evaluation :
#    A. Accuracy Score
#    B. Confusion Matrix
#    C. Classification Report

In [21]:
#Required Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [22]:
#Data Gathering
df = pd.read_csv("News_dataset.csv")
df.head()


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [23]:
#Data Analysis
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  72134 non-null  int64 
 1   title       71576 non-null  object
 2   text        72095 non-null  object
 3   label       72134 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 2.2+ MB


In [24]:
df['label'].value_counts()

label
1    37106
0    35028
Name: count, dtype: int64

In [25]:
df.shape

(72134, 4)

In [26]:
df.isna().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [27]:
df = df.dropna() #handled missing values by fropping these rows

In [28]:
df.isna().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [29]:
df.shape


(71537, 4)

In [30]:
df.reset_index(inplace = True)
df.head()


,index,Unnamed: 0,title,text,label
0,0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [31]:
df['title'][0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [32]:
df = df.drop(['Unnamed: 0', 'text'], axis=1, errors='ignore')
df.head()


,index,title,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
2,3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
3,4,SATAN 2: Russia unvelis an image of its terrif...,1
4,5,About Time! Christian Group Sues Amazon and SP...,1


In [33]:
#Data Preprocessing
#Tokenization
sample_data = 'The quick brown fox jumps over the lazy dog'
sample_data = sample_data.split()
sample_data



['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

In [34]:
# 2. Make Lowercase
sample_data = [data.lower() for data in sample_data]
sample_data

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

In [35]:
# 3. Remove Stopwords
import nltk
nltk.download('stopwords')

stopwords = stopwords.words('english')
print(stopwords[:10])
print(len(stopwords))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
198


[nltk_data] Downloading package stopwords to C:\Users\Kritika
[nltk_data]     dawar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
sample_data = [data for data in sample_data if data not in stopwords]
print(sample_data)
len(sample_data)

['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


6

In [37]:
# 4. Stemming
ps = PorterStemmer()
sample_data_stemming = [ps.stem(data) for data in sample_data]
print(sample_data_stemming)



['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']


In [40]:
# 5. Lemmatization
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lm = WordNetLemmatizer()
sample_data_lemma = [lm.lemmatize(data) for data in sample_data]
print(sample_data_lemma)

['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']


[nltk_data] Downloading package wordnet to C:\Users\Kritika
[nltk_data]     dawar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [41]:
lm = WordNetLemmatizer()
corpus = []
for i in range(len(df)):
    review = re.sub('[^a-zA-Z0-9 ]', '', df['title'][i])
    review = review.lower()
    review = review.split()
    review = [lm.lemmatize(x) for x in review if x not in stopwords]
    review = ' '.join(review)
    corpus.append(review)

In [42]:
len(corpus)

71537

In [44]:
df['title'][0]


'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [45]:
corpus[0]

'law enforcement high alert following threat cop white 911by blacklivesmatter fyf911 terrorist video'

In [48]:
#Vectorization(convert text data into vectors)
tf = TfidfVectorizer()
x = tf.fit_transform(corpus)
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 631674 stored elements and shape (71537, 32734)>

In [49]:
y = df['label']
y.head()

0    1
1    1
2    0
3    1
4    1
Name: label, dtype: int64

In [50]:
#Data splitting into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 10, stratify = y)


In [56]:
x_train.shape, y_train.shape


((50075, 32734), (50075,))

In [57]:
x_test.shape, y_test.shape

((21462, 32734), (21462,))

In [61]:
#ModelBuilding
rf = RandomForestClassifier(n_estimators=50, max_depth=10)
rf.fit(x_train, y_train)

,n_estimators,50
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [62]:
#ModelEvaluation
y_pred = rf.predict(x_test)
accuracy_score_ = accuracy_score(y_test, y_pred)
accuracy_score_

0.766051626129904

In [63]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

model = MultinomialNB()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.8713074270804212


In [64]:
tf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
x = tf.fit_transform(corpus)

In [65]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.8954431087503495


In [66]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

class Evaluation:

    def __init__(self, model, x_train, x_test, y_train, y_test):
        self.model = model
        self.x_train = x_train
        self.x_test = x_test
        self.y_train = y_train
        self.y_test = y_test

    def train_evaluation(self):
        y_pred_train = self.model.predict(self.x_train)

        acc_scr_train = accuracy_score(self.y_train, y_pred_train)
        print("Accuracy Score On Training Data Set :", acc_scr_train)

        con_mat_train = confusion_matrix(self.y_train, y_pred_train)
        print("Confusion Matrix On Training Data Set :\n", con_mat_train)

        class_rep_train = classification_report(self.y_train, y_pred_train)
        print("Classification Report On Training Data Set :\n", class_rep_train)


    def test_evaluation(self):
        y_pred_test = self.model.predict(self.x_test)

        acc_scr_test = accuracy_score(self.y_test, y_pred_test)
        print("Accuracy Score On Test Data Set :", acc_scr_test)

        con_mat_test = confusion_matrix(self.y_test, y_pred_test)
        print("Confusion Matrix On Test Data Set :\n", con_mat_test)

        class_rep_test = classification_report(self.y_test, y_pred_test)
        print("Classification Report On Test Data Set :\n", class_rep_test)

In [67]:
Evaluation(model, x_train, x_test, y_train, y_test).train_evaluation()

Accuracy Score On Training Data Set : 0.926510234648028
Confusion Matrix On Training Data Set :
 [[22314  2205]
 [ 1475 24081]]
Classification Report On Training Data Set :
               precision    recall  f1-score   support

           0       0.94      0.91      0.92     24519
           1       0.92      0.94      0.93     25556

    accuracy                           0.93     50075
   macro avg       0.93      0.93      0.93     50075
weighted avg       0.93      0.93      0.93     50075



In [68]:
Evaluation(model, x_train, x_test, y_train, y_test).test_evaluation()

Accuracy Score On Test Data Set : 0.8954431087503495
Confusion Matrix On Test Data Set :
 [[ 9218  1291]
 [  953 10000]]
Classification Report On Test Data Set :
               precision    recall  f1-score   support

           0       0.91      0.88      0.89     10509
           1       0.89      0.91      0.90     10953

    accuracy                           0.90     21462
   macro avg       0.90      0.90      0.90     21462
weighted avg       0.90      0.90      0.90     21462



In [69]:
#Prediction Pipeline
class Preprocessing:

    def __init__(self, data):
        self.data = data

    def text_preprocessing_user(self):
        lm = WordNetLemmatizer()
        pred_data = [self.data]
        preprocess_data = []

        for data in pred_data:
            review = re.sub('[^a-zA-Z0-9 ]', '', data)
            review = review.lower()
            review = review.split()
            review = [lm.lemmatize(x) for x in review if x not in stopwords]
            review = ' '.join(review)
            preprocess_data.append(review)

        return preprocess_data


In [70]:
df['title'][1]

'UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]'

In [71]:
data = 'UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]'
Preprocessing(data).text_preprocessing_user()


['unbelievable obamas attorney general say charlotte rioter peaceful protestersin home state north carolina video']

In [72]:
class Prediction:

    def __init__(self, pred_data, model):
        self.pred_data = pred_data
        self.model = model

    def prediction_model(self):
        preprocess_data = Preprocessing(self.pred_data).text_preprocessing_user()
        data = tf.transform(preprocess_data)
        prediction = self.model.predict(data)

        if prediction[0] == 0:
            return "The News Is Fake"
        else:
            return "The News Is Real"

In [76]:
# 1. Vectorization
tf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
x = tf.fit_transform(corpus)

# 2. Train-test split (VERY IMPORTANT AGAIN)
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=10, stratify=y
)

# 3. Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [77]:
data = "UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]"

result = Prediction(data, model).prediction_model()
print(result)

The News Is Real


In [79]:
import pickle

pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tf, open('tfidf.pkl', 'wb'))